# 🔍 03 — Model Evaluation & Error Analysis

Deep dive into model performance: confusion matrix, ROC curve,
error analysis, confidence distributions, and Grad-CAM visualization
on correct and incorrect predictions.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import torch

from pneumonia.utils.config import load_config
from pneumonia.utils.logging import setup_logging
from pneumonia.model.classifier import load_model
from pneumonia.data.dataset import create_dataloaders
from pneumonia.training.evaluator import Evaluator
from pneumonia.model.gradcam import GradCAMExplainer

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

setup_logging()

# Load model and data
config = load_config('../configs/train_config.yaml')
model = load_model(config.model, '../checkpoints/best_model.pth', device=config.device)
loaders = create_dataloaders(config)
evaluator = Evaluator(model, device=config.device)

CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

## 1. Confusion Matrix

In [ ]:
# Run predictions on test set
labels, probs, preds = evaluator.predict(loaders['test'])
metrics = evaluator.compute_metrics(labels, probs, preds)

# Plot confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, preds)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, ax=ax1, cbar=False)
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')
ax1.set_title('Confusion Matrix (Counts)', fontweight='bold')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, ax=ax2, cbar=False)
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')
ax2.set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.tight_layout()
plt.savefig('../evaluation/confusion_matrix_detailed.png', dpi=150, transparent=True)
plt.show()

# Print error breakdown
fp = cm[0, 1]  # Normal predicted as Pneumonia
fn = cm[1, 0]  # Pneumonia predicted as Normal (critical!)
print(f'\nFalse Positives (Normal → Pneumonia): {fp}')
print(f'False Negatives (Pneumonia → Normal): {fn} ← CRITICAL (missed pneumonia)')

## 2. ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(labels, probs)
auc = roc_auc_score(labels, probs)

ax1.plot(fpr, tpr, color='#1E88E5', linewidth=2, label=f'ROC (AUC = {auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax1.fill_between(fpr, tpr, alpha=0.1, color='#1E88E5')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve', fontweight='bold')
ax1.legend(fontsize=12)

# Precision-Recall Curve
precision_curve, recall_curve, thresholds_pr = precision_recall_curve(labels, probs)
ap = average_precision_score(labels, probs)

ax2.plot(recall_curve, precision_curve, color='#E53935', linewidth=2, label=f'PR (AP = {ap:.4f})')
ax2.fill_between(recall_curve, precision_curve, alpha=0.1, color='#E53935')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve', fontweight='bold')
ax2.legend(fontsize=12)

plt.tight_layout()
plt.savefig('../evaluation/roc_pr_curves.png', dpi=150, transparent=True)
plt.show()

## 3. Confidence Distribution

In [ ]:
# Analyze model confidence by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, cls in enumerate(CLASS_NAMES):
    mask = labels == idx
    cls_probs = probs[mask]
    
    axes[idx].hist(cls_probs, bins=30, color=['#4CAF50', '#F44336'][idx],
                  edgecolor='white', alpha=0.8)
    axes[idx].axvline(x=0.5, color='black', linestyle='--', alpha=0.5, label='Threshold')
    axes[idx].set_title(f'Confidence Distribution — {cls}', fontweight='bold')
    axes[idx].set_xlabel('P(Pneumonia)')
    axes[idx].set_ylabel('Count')
    axes[idx].legend()
    
    # Mark misclassified
    if cls == 'NORMAL':
        misclassified = cls_probs[cls_probs >= 0.5]
    else:
        misclassified = cls_probs[cls_probs < 0.5]
    print(f'{cls}: {len(misclassified)} misclassified out of {len(cls_probs)}')

plt.tight_layout()
plt.savefig('../evaluation/confidence_distribution.png', dpi=150, transparent=True)
plt.show()

## 4. Error Analysis — Misclassified Images

In [ ]:
# Collect test set file paths
test_dir = Path(config.data.root) / 'test'
test_files = []
test_labels_ordered = []

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_dir = test_dir / cls_name
    files = sorted(cls_dir.glob('*'))
    test_files.extend(files)
    test_labels_ordered.extend([cls_idx] * len(files))

# Note: DataLoader may shuffle differently, so we re-run prediction
# on individual files to match paths to predictions
from pneumonia.inference.predictor import Predictor

predictor = Predictor(model, image_size=config.data.image_size,
                     device=config.device, enable_gradcam=False)

# Find misclassified images
errors = []
for img_path, true_label in zip(test_files, test_labels_ordered):
    result = predictor.predict(str(img_path))
    pred_label = result['class_index']
    if pred_label != true_label:
        errors.append({
            'path': img_path,
            'true': CLASS_NAMES[true_label],
            'predicted': result['label'],
            'confidence': result['confidence'],
            'prob_pneumonia': result['probability_pneumonia'],
        })

print(f'Total misclassified: {len(errors)} / {len(test_files)} ({len(errors)/len(test_files)*100:.1f}%)')
print(f'\nFalse Negatives (Pneumonia → Normal): {sum(1 for e in errors if e["true"]=="PNEUMONIA")}')
print(f'False Positives (Normal → Pneumonia): {sum(1 for e in errors if e["true"]=="NORMAL")}')

In [ ]:
# Visualize misclassified images
n_show = min(10, len(errors))
if n_show > 0:
    cols = 5
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.array(axes).flatten()

    for i, err in enumerate(errors[:n_show]):
        img = Image.open(err['path']).convert('L')
        axes[i].imshow(img, cmap='gray')
        color = 'red' if err['true'] == 'PNEUMONIA' else 'orange'
        axes[i].set_title(
            f"True: {err['true']}\nPred: {err['predicted']} ({err['confidence']:.1%})",
            fontsize=9, color=color, fontweight='bold'
        )
        axes[i].axis('off')

    # Hide unused subplots
    for j in range(n_show, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Misclassified Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../evaluation/misclassified_images.png', dpi=150, bbox_inches='tight', transparent=True)
    plt.show()
else:
    print('No misclassified images! 🎉')

## 5. Grad-CAM on Correct vs. Incorrect Predictions

In [ ]:
# Grad-CAM explainer
explainer = GradCAMExplainer(model, image_size=config.data.image_size, device=config.device)

def show_gradcam_grid(image_paths, titles, ncols=4):
    """Display Grad-CAM for a list of images."""
    n = len(image_paths)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()
    
    for i, (path, title) in enumerate(zip(image_paths, titles)):
        _, overlay = explainer.explain(str(path))
        axes[i].imshow(overlay)
        axes[i].set_title(title, fontsize=9, fontweight='bold')
        axes[i].axis('off')
    
    for j in range(n, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    return fig

In [ ]:
# Grad-CAM on correctly classified samples
correct_normal = [f for f, l in zip(test_files, test_labels_ordered)
                  if l == 0 and f not in [e['path'] for e in errors]][:4]
correct_pneumonia = [f for f, l in zip(test_files, test_labels_ordered)
                     if l == 1 and f not in [e['path'] for e in errors]][:4]

print('Grad-CAM on CORRECT Normal predictions:')
fig = show_gradcam_grid(correct_normal, ['Normal ✅'] * len(correct_normal))
fig.savefig('../evaluation/gradcam_correct_normal.png', dpi=150, bbox_inches='tight', transparent=True)
plt.show()

print('\nGrad-CAM on CORRECT Pneumonia predictions:')
fig = show_gradcam_grid(correct_pneumonia, ['Pneumonia ✅'] * len(correct_pneumonia))
fig.savefig('../evaluation/gradcam_correct_pneumonia.png', dpi=150, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# Grad-CAM on misclassified samples
if errors:
    error_paths = [e['path'] for e in errors[:8]]
    error_titles = [f"True: {e['true']} → Pred: {e['predicted']}" for e in errors[:8]]
    
    print('Grad-CAM on MISCLASSIFIED predictions:')
    fig = show_gradcam_grid(error_paths, error_titles)
    fig.savefig('../evaluation/gradcam_errors.png', dpi=150, bbox_inches='tight', transparent=True)
    plt.show()
else:
    print('No errors to visualize!')

## 6. Threshold Sensitivity Analysis

In [ ]:
# How do metrics change with different decision thresholds?
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

thresholds = np.arange(0.1, 0.95, 0.05)
results = {'threshold': [], 'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

for t in thresholds:
    t_preds = (probs >= t).astype(int)
    results['threshold'].append(t)
    results['accuracy'].append(accuracy_score(labels, t_preds))
    results['precision'].append(precision_score(labels, t_preds, zero_division=0))
    results['recall'].append(recall_score(labels, t_preds, zero_division=0))
    results['f1'].append(f1_score(labels, t_preds, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(results['threshold'], results['accuracy'], 'o-', label='Accuracy', linewidth=2)
ax.plot(results['threshold'], results['precision'], 's-', label='Precision', linewidth=2)
ax.plot(results['threshold'], results['recall'], '^-', label='Recall', linewidth=2)
ax.plot(results['threshold'], results['f1'], 'D-', label='F1', linewidth=2)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')

ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Threshold Sensitivity Analysis', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig('../evaluation/threshold_analysis.png', dpi=150, transparent=True)
plt.show()

# Find optimal threshold for recall >= 0.95
for i, t in enumerate(results['threshold']):
    if results['recall'][i] >= 0.95:
        print(f'\nHighest threshold with recall ≥ 95%: {t:.2f}')
        print(f'  Accuracy:  {results["accuracy"][i]:.4f}')
        print(f'  Precision: {results["precision"][i]:.4f}')
        print(f'  Recall:    {results["recall"][i]:.4f}')
        print(f'  F1:        {results["f1"][i]:.4f}')
        break

## 7. Key Findings

- **False negatives** (missed pneumonia) are the most critical errors — review Grad-CAM on those cases
- **Grad-CAM quality**: Correct pneumonia predictions should highlight lung opacity regions
- **Threshold 0.5** works well, but lowering it could further reduce false negatives at the cost of more false positives
- **Confidence calibration**: Most predictions are high-confidence, indicating the model is decisive
- **Error patterns**: Look for common characteristics in misclassified images (equipment artifacts, edge cases)